<a href="https://colab.research.google.com/github/Quantum-Hobyist/quantum-realism-demos/blob/main/demos/01%20tiefste_energiezustaende_Li_und_H2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-nature qiskit-algorithms pyscf -q

In [4]:
# === Demo 1: VQE in der Wirkstoffforschung – funktioniert, aber skaliert nicht ===
# Colab: zuerst in einer eigenen Zelle installieren:
#   !pip install qiskit qiskit-nature qiskit-algorithms pyscf -q

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer, FreezeCoreTransformer
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

def run_vqe(problem, label):
    mapper = ParityMapper(num_particles=problem.num_particles)
    ansatz = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper,
        initial_state=HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper))
    vqe = VQE(StatevectorEstimator(), ansatz, SLSQP())
    vqe.initial_point = [0.0] * ansatz.num_parameters
    e_vqe   = GroundStateEigensolver(mapper, vqe).solve(problem).total_energies[0].real
    e_exact = GroundStateEigensolver(mapper, NumPyMinimumEigensolver()).solve(problem).total_energies[0].real
    print(f"{label}: {ansatz.num_qubits} Qubits, {ansatz.num_parameters} Parameter")
    print(f"   VQE = {e_vqe:.6f} Hartree | Exakt = {e_exact:.6f} Hartree\n")

# Stufe 1 – H2: läuft voll, trifft exakt, in Sekunden
h2 = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto3g").run()
run_vqe(h2, "H2 (voll)")

# Stufe 2 – LiH: nur mit kleinem Aktivraum schnell lösbar
lih = ActiveSpaceTransformer(num_electrons=2, num_spatial_orbitals=2).transform(
        PySCFDriver(atom="Li 0 0 0; H 0 0 1.6", basis="sto3g").run())
run_vqe(lih, "LiH (Aktivraum 2e,2o)")

# Stufe 3 – Skalierung: was die vollen Moleküle bräuchten (nur zählen, kein VQE)
print("Skalierung (Rumpf eingefroren, kein VQE):")
for name, atom in [("H2","H 0 0 0; H 0 0 0.735"),
                   ("LiH","Li 0 0 0; H 0 0 1.6"),
                   ("BeH2","Be 0 0 0; H 0 0 1.3; H 0 0 -1.3"),
                   ("H2O","O 0 0 0; H 0 0.76 -0.48; H 0 -0.76 -0.48")]:
    p = FreezeCoreTransformer().transform(PySCFDriver(atom=atom, basis="sto3g").run())
    m = ParityMapper(num_particles=p.num_particles)
    a = UCCSD(p.num_spatial_orbitals, p.num_particles, m,
        initial_state=HartreeFock(p.num_spatial_orbitals, p.num_particles, m))
    print(f"   {name:5s} {a.num_qubits:2d} Qubits, {a.num_parameters:3d} Parameter")



H2 (voll): 2 Qubits, 3 Parameter
   VQE = -1.137306 Hartree | Exakt = -1.137306 Hartree

LiH (Aktivraum 2e,2o): 2 Qubits, 3 Parameter
   VQE = -7.862129 Hartree | Exakt = -7.862129 Hartree

Skalierung (Rumpf eingefroren, kein VQE):
   H2     2 Qubits,   3 Parameter
   LiH    8 Qubits,  24 Parameter
   BeH2  10 Qubits,  92 Parameter
   H2O   10 Qubits,  92 Parameter


Funktioniert ein quantenunterstützter, hybrider Ansatz zur Bestimmung des tiefsten molekularen Energiezustandes (VQE) in der Wirkstoffforschung wirklich? Eine nachvollziehbare Demo darüber, ob Quantum Pharmaunternehmen tatsächlich hilft und ob das Ganze bereits skalierbar ist.

Zuerst wurde die Methode am simplen Beispiel eines H2 Moleküls getestet. Der VQE Ansatz hat dort das richtige Ergebnis geliefert, nämlich −1.137306 Hartree (Einheit der atomaren Quantenchemie ca. 27.2 eV) Das Ergebnis konnte dort innerhalb weniger Sekunden ermittelt werden. VQE brauchte 2 Qubits.

Anschliessend wurde ein komplizierteres Molekül getestet, das LiH Molekül. Hier war das Verfahren deutlich komplexer. Auf herkömmlichem Weg, mit dem ganzen Molekül konnte die Grundzustandsenergie selbst nach über 10min nicht von VQE berechnet werden. Dies führte zur Entscheidung, den Atomrumpf einzufrieren, um das Problem auf machbare Grösse herunterzubrechen. So konnte auch da innerhalb einiger Minuten ein exakter Wert von -7.882096599921165 Hartree (8 Qubits) ermittelt werden. Da dies aber immer noch zu lange dauerte wurde das Molekül auf einen Aktivraum von 2 Qubits weiter heruntergebrochen. Dort kam die Lösung innert Sekunden allerdings mit leicht verändertem Wert nämlich -7.862129 Hartree. Dieses Beispiel zeigt jedoch eindrücklich, dass VQE nur in kleineren Testumgebungen funktionsfähig ist und noch nicht auf dem realen Markt.

In der folgenden Tabelle werden die mit zunehmender Grösse des Moleküls schnell ansteigenden Werte der Parameter und Qubits gezeigt
    H2     2 Qubits,   3 Parameter
    LiH    8 Qubits,  24 Parameter
    BeH2  10 Qubits,  92 Parameter
    H2O   10 Qubits,  92 Parameter

Zusammenfassend lässt sich also sagen, dass VQE die richtigen Werte exakt trifft, aber nur dann, wenn die Moleküle simpel genug sind. Die Vorteile von Quantum in der Wirkstoffforschung können realisiert werden, doch dafür braucht es bessere Hardware
